In [2]:
"""
04_baseline_model_benchmark.ipynb

Purpose:
Train and compare baseline classification models on the train split
and evaluate them on the validation split for fair model selection.

Models included:
1. Random Forest
2. XGBoost
3. LightGBM

What this notebook does:
1. Load train and validation splits
2. Train baseline models
3. Evaluate using multi-class metrics
4. Compare models fairly on validation data
5. Save only the summary files needed for viva and later selection

"""

# 1. Imports

# sys is used only to print Python version
import sys

# Path helps build file paths clearly
from pathlib import Path

# warnings are hidden to keep notebook output cleaner
import warnings

# numpy and pandas are used for data handling
import numpy as np
import pandas as pd

# Metrics used to compare models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)

# Used to convert multiclass labels for ROC-AUC calculation
from sklearn.preprocessing import label_binarize

# Baseline model 1
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# display() looks nicer in notebooks, but print() is used if display is unavailable
try:
    from IPython.display import display
except Exception:
    display = None

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


# 2. Optional Model Imports

# XGBoost and LightGBM may not always be installed
HAS_XGBOOST = True
HAS_LIGHTGBM = True

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGBOOST = False
    print("XGBoost not available. Skipping XGBClassifier.")

try:
    from lightgbm import LGBMClassifier
except Exception:
    HAS_LIGHTGBM = False
    print("LightGBM not available. Skipping LGBMClassifier.")


# 3. Paths

PROJECT_ROOT = Path.cwd().parent
SPLITS_PATH = PROJECT_ROOT / "data" / "splits"
REPORTS_PATH = PROJECT_ROOT / "reports" / "baseline_benchmark"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SPLITS_PATH  :", SPLITS_PATH)
print("REPORTS_PATH :", REPORTS_PATH)


# 4. Helper Functions

def show_df(df: pd.DataFrame, n: int = 5) -> None:
    """
    Show the first few rows of a dataframe.
    """
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n))


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Save dataframe as CSV.
    """
    df.to_csv(path, index=index)


def evaluate_multiclass(y_true, y_pred, y_proba, class_labels):
    """
    Compute the main multi-class evaluation metrics.

    Why these metrics?
    - Accuracy shows overall correctness
    - Balanced accuracy is useful for imbalanced classes
    - Macro precision/recall/F1 treat all classes equally
    - Macro F1 is the main metric for model selection
    - ROC-AUC OVR gives a probability-based view of performance
    """
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    # Convert multiclass labels into binary indicator form for ROC-AUC
    y_true_bin = label_binarize(y_true, classes=class_labels)

    result["roc_auc_ovr_macro"] = roc_auc_score(
        y_true_bin,
        y_proba,
        multi_class="ovr",
        average="macro",
    )

    result["roc_auc_ovr_weighted"] = roc_auc_score(
        y_true_bin,
        y_proba,
        multi_class="ovr",
        average="weighted",
    )

    return result


def get_model_dict():
    """
    Create the baseline models to compare.

    These are simple baseline versions, not tuned versions.
    """
    models = {}

    # Random Forest baseline
    models["random_forest"] = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    # XGBoost baseline
    if HAS_XGBOOST:
        models["xgboost"] = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
        )

    # LightGBM baseline
    if HAS_LIGHTGBM:
        models["lightgbm"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

    return models


def detect_group(col: str) -> str:
    """
    Detect which feature group a column belongs to.
    This is only for the feature group summary report.
    """
    if col == "location_num":
        return "location"
    if col.startswith("severity_type_"):
        return "severity_type"
    if col.startswith("event_type_"):
        return "event_type"
    if col.startswith("resource_type_"):
        return "resource_type"
    if col.startswith("log_feature_") and col.endswith("_vol"):
        return "log_feature_volume"
    if col.startswith("log_feature_"):
        return "log_feature_presence"
    return "engineered"


# 5. Load Splits

# Load train and validation feature sets
X_train = pd.read_csv(SPLITS_PATH / "X_train.csv")
X_valid = pd.read_csv(SPLITS_PATH / "X_valid.csv")

# Load train and validation labels
y_train = pd.read_csv(SPLITS_PATH / "y_train.csv")["fault_severity"]
y_valid = pd.read_csv(SPLITS_PATH / "y_valid.csv")["fault_severity"]

print("\nLoaded splits:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

# Make sure feature columns match between train and validation
if list(X_train.columns) != list(X_valid.columns):
    raise ValueError("Train/validation feature mismatch.")

# Get sorted class labels
class_labels = sorted(y_train.unique().tolist())
print("Classes:", class_labels)


# 6. Baseline Training and Evaluation

# Create baseline models
models = get_model_dict()

if len(models) == 0:
    raise ValueError("No models available to benchmark.")

benchmark_rows = []
per_class_rows = []

# Train and evaluate each model
for model_name, model in models.items():
    print(f"\nTRAINING: {model_name}")

    # Fit model on training split
    model.fit(X_train, y_train)

    # Predict classes and class probabilities on validation split
    y_pred = model.predict(X_valid)
    y_proba = model.predict_proba(X_valid)

    # Compute summary metrics
    metrics = evaluate_multiclass(
        y_true=y_valid,
        y_pred=y_pred,
        y_proba=y_proba,
        class_labels=class_labels,
    )

    # Store one summary row per model
    benchmark_row = {"model": model_name}
    benchmark_row.update({k: round(v, 6) for k, v in metrics.items()})
    benchmark_rows.append(benchmark_row)

    # Build per-class metrics table
    clf_rep = classification_report(
        y_valid,
        y_pred,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )

    for cls in class_labels:
        cls_key = str(cls)
        per_class_rows.append(
            {
                "model": model_name,
                "class": cls,
                "precision": clf_rep[cls_key]["precision"],
                "recall": clf_rep[cls_key]["recall"],
                "f1_score": clf_rep[cls_key]["f1-score"],
                "support": clf_rep[cls_key]["support"],
            }
        )

    print(f"{model_name} done.")


# 7. Benchmark Summary Table

# Create model comparison table
benchmark_df = pd.DataFrame(benchmark_rows)

# Sort by the main model selection rule
benchmark_df = benchmark_df.sort_values(
    by=["f1_macro", "balanced_accuracy", "roc_auc_ovr_macro"],
    ascending=False,
).reset_index(drop=True)

print("\nBENCHMARK SUMMARY")
print(benchmark_df)

# Save overall benchmark summary
save_csv(benchmark_df, REPORTS_PATH / "baseline_benchmark_summary.csv", index=False)

# Save per-class metrics
per_class_df = pd.DataFrame(per_class_rows)
save_csv(per_class_df, REPORTS_PATH / "baseline_per_class_metrics.csv", index=False)

if display is not None:
    display(benchmark_df)
    display(per_class_df.head(12))


# 8. Best Baseline Model

# The first row after sorting is the best baseline model
best_model_name = benchmark_df.iloc[0]["model"]
print("\nBest baseline model by Macro F1:", best_model_name)

best_model_summary = pd.DataFrame(
    {
        "selection_rule": ["Highest Macro F1 on validation set"],
        "best_model": [best_model_name],
        "f1_macro": [benchmark_df.iloc[0]["f1_macro"]],
        "balanced_accuracy": [benchmark_df.iloc[0]["balanced_accuracy"]],
        "roc_auc_ovr_macro": [benchmark_df.iloc[0]["roc_auc_ovr_macro"]],
    }
)

save_csv(best_model_summary, REPORTS_PATH / "best_baseline_model_summary.csv", index=False)


# 9. Feature Group Summary

# Build feature group breakdown
feature_names = X_train.columns.tolist()

feature_group_df = pd.DataFrame({"feature_name": feature_names})
feature_group_df["feature_group"] = feature_group_df["feature_name"].apply(detect_group)

feature_group_summary = (
    feature_group_df["feature_group"]
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="count")
)

print("\nFEATURE GROUP SUMMARY")
print(feature_group_summary)

save_csv(feature_group_summary, REPORTS_PATH / "feature_group_summary.csv", index=False)


# 10. Compact Final Summary

# Small summary table 
final_summary = pd.DataFrame(
    {
        "item": [
            "n_train_rows",
            "n_valid_rows",
            "n_features",
            "n_models_benchmarked",
            "primary_metric",
            "best_baseline_model",
        ],
        "value": [
            len(X_train),
            len(X_valid),
            X_train.shape[1],
            len(models),
            "f1_macro",
            best_model_name,
        ],
    }
)

print("\nFINAL SUMMARY")
print(final_summary)

save_csv(final_summary, REPORTS_PATH / "baseline_notebook_summary.csv", index=False)

print("\nBASELINE BENCHMARKING COMPLETE")
print("Reports saved to :", REPORTS_PATH)

Python : 3.12.2
Pandas : 2.3.3
NumPy  : 2.3.5
PROJECT_ROOT : /Users/hasheenadilmidesilva/Desktop/NetGuard
SPLITS_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/splits
REPORTS_PATH : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/baseline_benchmark

Loaded splits:
X_train: (4723, 860)
X_valid: (1181, 860)
y_train: (4723,)
y_valid: (1181,)
Classes: [0, 1, 2]

TRAINING: random_forest
random_forest done.

TRAINING: xgboost
xgboost done.

TRAINING: lightgbm
lightgbm done.

BENCHMARK SUMMARY
           model  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro  precision_weighted  recall_weighted  f1_weighted  roc_auc_ovr_macro  roc_auc_ovr_weighted
0       lightgbm  0.743438           0.757010         0.683486      0.757010  0.709218            0.779236         0.743438     0.752385           0.884613              0.863387
1        xgboost  0.763760           0.698254         0.708979      0.698254  0.703078            0.759094         0.763760     0.760913

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_macro,roc_auc_ovr_weighted
0,lightgbm,0.743438,0.757010,0.683486,0.757010,0.709218,0.779236,0.743438,0.752385,0.884613,0.863387
1,xgboost,0.763760,0.698254,0.708979,0.698254,0.703078,0.759094,0.763760,0.760913,0.891408,0.872118
2,random_forest,0.743438,0.658108,0.686406,0.658108,0.670730,0.735648,0.743438,0.738337,0.864839,0.841804


,model,class,precision,recall,f1_score,support
0,random_forest,0,0.803198,0.852480,0.827106,766.0
1,random_forest,1,0.589354,0.518395,0.551601,299.0
2,random_forest,2,0.666667,0.603448,0.633484,116.0
3,xgboost,0,0.823678,0.853786,0.838462,766.0
4,xgboost,1,0.625000,0.568562,0.595447,299.0
5,xgboost,2,0.678261,0.672414,0.675325,116.0
6,lightgbm,0,0.892913,0.740209,0.809422,766.0
7,lightgbm,1,0.557545,0.729097,0.631884,299.0
8,lightgbm,2,0.600000,0.801724,0.686347,116.0



Best baseline model by Macro F1: lightgbm

FEATURE GROUP SUMMARY
          feature_group  count
0  log_feature_presence    386
1    log_feature_volume    386
2            event_type     53
3            engineered     18
4         resource_type     10
5         severity_type      6
6              location      1

FINAL SUMMARY
                   item     value
0          n_train_rows      4723
1          n_valid_rows      1181
2            n_features       860
3  n_models_benchmarked         3
4        primary_metric  f1_macro
5   best_baseline_model  lightgbm

BASELINE BENCHMARKING COMPLETE
Reports saved to : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/baseline_benchmark
